<a href="https://colab.research.google.com/github/DeniseValeriaVelarde/Edge-LLM-Benchmark-RaspberryPi5/blob/main/BenchmarkSimples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import time
import pandas as pd
from llama_cpp import Llama
import psutil

#Configuração dos caminhos
HOME_DIR = os.path.expanduser("~")
LLAMA_CPP_DIR = os.path.join(HOME_DIR, "llama.cpp")
MODELS_DIR = os.path.join(HOME_DIR, "models")
DATASET_PATH = os.path.join(HOME_DIR, "dataset.csv")
RESULTS_DIR = os.path.join(HOME_DIR, "results")


def save_results(results_list, model_name):

    if not results_list:
        print("Nenhuma lista de resultados para salvar.")
        return

    if not os.path.exists(RESULTS_DIR):
        print(f"Criando diretório de resultados em: {RESULTS_DIR}")
        os.makedirs(RESULTS_DIR)

    #cria um nome de arquivo, removendo extensões e caracteres especiais.
    safe_model_name = model_name.replace('.gguf', '').replace('.', '_')
    output_path = os.path.join(RESULTS_DIR, f"results_{safe_model_name}.csv")

    #converte a lista de resultados em Pandas e salva em csv.
    df = pd.DataFrame(results_list)
    df.to_csv(output_path, index=False)

    print(f"\n Resultados salvos com sucesso em: {output_path}")

def run_benchmark(model_name, dataset):
    model_path = os.path.join(MODELS_DIR, model_name)
    if not os.path.exists(model_path):
        print(f"Erro: Arquivo de modelo não encontrado em: {model_path}")
        return []

    #Medição de RAM
    #Pega o processo atual para monitorar o uso de memória RAM.
    process = psutil.Process(os.getpid())
    peak_ram_mb = 0

    results = []

    try:
        #carrega o modelo uma unica vez
        print(f"Carregando o modelo '{model_name}'... ")
        llm = Llama(
            model_path=model_path,
            n_ctx=512,      #tokens utilizados
            n_threads=4,    #threads da CPU
            n_gpu_layers=0, #roda exclusivamente na CPU
            verbose=False
        )
        print("Modelo carregado")

        #medicao da ram
        ram_after_load = process.memory_info().rss / (1024 * 1024)
        peak_ram_mb = ram_after_load
        print(f"RAM usada após carregar o modelo: {ram_after_load:.2f} MB")

        #itera sobre o dataset e coleta as metricas que quero
        for i, row in enumerate(dataset):
            prompt_text = row['frase']
            prompt_full = f"Classifique a frase: '{prompt_text}' com uma das seguintes intenções: LIGAR_LUZ, DESLIGAR_LUZ, AJUSTAR_TEMPERATURA, TOCAR_MUSICA, OBTER_HORA ou NAO_RECONHECIDO, intenção é:"

            print(f"Processando frase {i+1}/{len(dataset)}: '{prompt_text}'")

            start_time = time.time()

            #aqui temos a geracao dos resultados
            output = llm(
                prompt_full,
                max_tokens=10,
                temperature=0.2,    #baixa temperatura pra não ser tão criativo
                stop=["\n", "Frase:", "Intenção:"],
                echo=False
            )

            end_time = time.time()

            #extração e calculo das metricas
            answer_text = output['choices'][0]['text'].strip()
            generated_answer = answer_text.split()[0] if answer_text else "VAZIO"

            user_latency_s = end_time - start_time
            tokens_generated = output['usage']['completion_tokens']
            tokens_per_sec = tokens_generated / user_latency_s if user_latency_s > 0 else 0

            #pico de RAM
            current_ram_mb = process.memory_info().rss / (1024 * 1024)
            if current_ram_mb > peak_ram_mb:
                peak_ram_mb = current_ram_mb

            results.append({
                "model_name": model_name,
                "prompt": prompt_text,
                "generated_answer": generated_answer,
                "expected_intention": row['intencao'],
                "latency_ms": user_latency_s * 1000,
                "tokens_per_second": tokens_per_sec
            })

        # adiciona o pico a todos os registros para análise posterior
        for res in results:
            res['peak_ram_mb'] = peak_ram_mb

    except Exception as e:
        print(f"Ocorreu um erro inesperado durante o benchmark: {e}")
        return results

    return results

if __name__ == "__main__":
    # verifica se o nome do modelo foi passado como argumento
    if len(sys.argv) < 2:
        print(f"Uso: python3 {sys.argv[0]} <nome_do_arquivo_do_modelo.gguf>")
        sys.exit(1)

    target_model_name = sys.argv[1]

    print("Iniciando...")
    print(f"Modelo Alvo: {target_model_name}")

    # Carrega o dataset
    try:
        print(f"Carregando dataset de '{DATASET_PATH}'...")
        dataset_records = pd.read_csv(DATASET_PATH).to_dict('records')
        print(f"Dataset carregado com {len(dataset_records)} frases.")
    except FileNotFoundError:
        print(f"Erro: Dataset não encontrado em '{DATASET_PATH}'")
        sys.exit(1)

    if not dataset_records:
        print("dataset está vazio.")
        sys.exit(1)

    #executa o benchmark
    all_results = run_benchmark(target_model_name, dataset_records)

    #salva resultados
    if all_results:
        save_results(all_results, target_model_name)
    else:
        print("Nenhum resultado foi gerado para salvar.")

    print("FIM")
